<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/11-Adding_Hybrid_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid Search: Combining Vector and Keyword Retrieval

This notebook implements **hybrid search** by combining vector (semantic) retrieval with keyword-based retrieval using LlamaIndex. Hybrid search often outperforms either approach alone, especially for queries containing proper nouns, version numbers, or technical terms that semantic search may under-weight.

## Learning Objectives

By the end of this notebook, you will be able to:
- Load a pre-built vector store from HuggingFace Hub
- Build a `SimpleKeywordTableIndex` from the same document nodes
- Implement a custom `HybridRetriever` that interleaves vector and keyword results
- Evaluate hybrid vs. vector-only retrieval using `RetrieverEvaluator` (Hit Rate, MRR)

## Prerequisites

- Familiarity with LlamaIndex `VectorStoreIndex` and retrievers
- Basic understanding of RAG pipelines
- OpenAI API key (set in Colab Secrets as `OPENAI_API_KEY`)

## Install Packages and Setup Variables


In [ ]:
!pip install -qU llama-index==0.14.16 openai==2.30.0 google-genai==1.70.0 llama-index-embeddings-openai==0.5.2 \
                chromadb==1.5.5 llama-index-vector-stores-chroma==0.5.5 nest_asyncio jedi==0.19.2 opentelemetry-api==1.38.0\
                llama-index-llms-openai==0.6.26 llama-index-finetuning==0.4.2 llama-index-embeddings-huggingface==0.7.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# !pip install -q llama-index==0.14.19 openai==2.8.1 google-genai==1.70.0 llama-index-embeddings-openai==0.6.0 opentelemetry-api==1.38.0 \
#                 chromadb==1.5.5 llama-index-vector-stores-chroma==0.5.5 nest_asyncio jedi==0.19.2 \
#                 llama-index-llms-openai==0.7.5 llama-index-finetuning==0.4.2 llama-index-embeddings-huggingface==0.7.0

In [ ]:
import os

# Load the OpenAI API key from Colab Secrets, with a local environment variable fallback.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except ImportError:
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")

In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.

import nest_asyncio

nest_asyncio.apply()

# Initialize Models

In [ ]:
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAI(model="gpt-5-mini", additional_kwargs={"reasoning_effort": "low"})
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Download knowledge base


In [ ]:
from huggingface_hub import hf_hub_download

hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore.zip", repo_type="dataset", local_dir=".")

vectorstore.zip:   0%|          | 0.00/97.2M [00:00<?, ?B/s]

'vectorstore.zip'

In [ ]:
!unzip -o vectorstore.zip

Archive:  vectorstore.zip
   creating: ai_tutor_knowledge/
   creating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/length.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/index_metadata.pickle  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/link_lists.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/header.bin  
  inflating: ai_tutor_knowledge/684af133-f877-4230-bde4-575cf53b6688/data_level0.bin  
  inflating: ai_tutor_knowledge/chroma.sqlite3  


# Create Vector Index from Persisted Chroma Store

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex

# Load the vector store from the local storage.
db = chromadb.PersistentClient(path="./ai_tutor_knowledge")
chroma_collection = db.get_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Create the index based on the vector store.
vector_index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

# Create Keyword Index from the Same Document Nodes

In [ ]:
def retrieve_all_nodes_from_vector_index(vector_index, query="Whatever", similarity_top_k=100000000):
    # Set similarity_top_k to a large number to retrieve all the nodes
    vector_retriever = vector_index.as_retriever(similarity_top_k=similarity_top_k)

    # Retrieve all nodes
    all_nodes = vector_retriever.retrieve(query)
    nodes = [item.node for item in all_nodes]

    return nodes

nodes = retrieve_all_nodes_from_vector_index(vector_index)
print(len(nodes))

5834


In [ ]:
from llama_index.core import SimpleKeywordTableIndex

# Build the keyword table index from all the document nodes.
keyword_index = SimpleKeywordTableIndex(nodes=nodes)

## Optional: What Does the Keyword Index Store?

In [ ]:
# Inspect what keywords are indexed in the SimpleKeywordTableIndex
try:
    keyword_table = keyword_index.index_struct
    all_keywords = list(keyword_table.table.keys())
    print(f"Total unique keywords indexed: {len(all_keywords)}")
    print(f"\nSample keywords (first 30):")
    print(", ".join(sorted(all_keywords)[:30]))
    print(f"\nThese keywords are matched exactly against the query,")
    print(f"complementing the semantic matching from the vector index.")
except Exception as e:
    print(f"Could not inspect keyword index: {e}")

Total unique keywords indexed: 5034

Sample keywords (first 30):
0, 00, 000, 000000, 002, 00875f, 01, 0202, 0301, 04, 07, 0alternative, 0m, 0removal, 1, 10, 100, 1000, 101, 1024, 11, 12, 120, 128, 13b, 14, 1401, 15, 150, 17

These keywords are matched exactly against the query,
complementing the semantic matching from the vector index.


# Hybrid Retriever


In [ ]:
from llama_index.core import QueryBundle
from llama_index.core.schema import NodeWithScore
from llama_index.core.retrievers import (
    BaseRetriever,
    VectorIndexRetriever,
    KeywordTableSimpleRetriever,
)
from typing import List

class HybridRetriever(BaseRetriever):
    """Hybrid retriever that performs both semantic search and keyword search."""

    def __init__(
        self,
        vector_retriever: VectorIndexRetriever,
        keyword_retriever: KeywordTableSimpleRetriever,
        max_retrieve: int = 10,
    ) -> None:
        """Init params."""

        self._vector_retriever = vector_retriever
        self._keyword_retriever = keyword_retriever
        self._max_retrieve = max_retrieve
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        """Retrieve nodes given query."""

        vector_nodes = self._vector_retriever.retrieve(query_bundle)
        keyword_nodes = self._keyword_retriever.retrieve(query_bundle)

        resulting_nodes = []
        node_ids_added = set()
        for i in range(min(len(vector_nodes), len(keyword_nodes))):
            vector_node = vector_nodes[i]
            if vector_node.node.node_id not in node_ids_added:
                resulting_nodes += [vector_node]
                node_ids_added.add(vector_node.node.node_id)

            keyword_node = keyword_nodes[i]
            if keyword_node.node.node_id not in node_ids_added:
                resulting_nodes += [keyword_node]
                node_ids_added.add(keyword_node.node.node_id)

        return resulting_nodes

# Test hybrid retriever vs vector retriever

In [ ]:
from llama_index.core import get_response_synthesizer
from llama_index.core.query_engine import RetrieverQueryEngine

# Create hybrid query engine
vector_retriever = VectorIndexRetriever(index=vector_index, similarity_top_k=6)
keyword_retriever = KeywordTableSimpleRetriever(index=keyword_index, num_chunks_per_query=6)
hybrid_retriever = HybridRetriever(vector_retriever, keyword_retriever, max_retrieve=6)
response_synthesizer = get_response_synthesizer(llm=Settings.llm)
hybrid_query_engine = RetrieverQueryEngine(
    retriever=hybrid_retriever,
    response_synthesizer=response_synthesizer,
)

# Test the query engine
answer = hybrid_query_engine.query("How does KOSMOS-2 work?")
print(answer)

KOSMOS-2 is a multimodal Transformer-based causal language model that learns to predict the next token on large-scale grounded image–text data. Its key mechanisms and workflow:

- Training objective and data
  - Trained with next-token prediction on a web-scale dataset of grounded image–text pairs.
  - Grounding information (image regions / bounding boxes) is included in the training data so the model learns to link text spans to visual regions.

- Representation of visual grounding
  - Spatial coordinates of bounding boxes are converted into sequences of discrete location tokens (e.g., patch indices).
  - These location-token sequences are appended to the corresponding entity text spans so that object mentions and their image regions are represented together (a hyperlink-like pairing).
  - Referential expressions are encoded in a Markdown-like link form (e.g., [text span](bounding boxes)), enabling explicit text↔region links.

- Multimodal input and generation
  - Inputs combine text 

In [ ]:
# Create vector query engine
vector_retriever = VectorIndexRetriever(index=vector_index, similarity_top_k=6)
vector_query_engine = RetrieverQueryEngine(
    retriever=vector_retriever,
    response_synthesizer=response_synthesizer,
)

# Test the query engine
answer = vector_query_engine.query("How does KOSMOS-2 work?")
print(answer)

KOSMOS-2 is a multimodal Transformer-style causal language model that grounds language to visual regions and is trained with next-word prediction on a large corpus of image–text pairs. Key aspects of how it works:

- Multimodal input and training: images and associated captions/annotations are used together so the model learns to predict text conditioned on visual content.
- Spatial grounding via location tokens: bounding-box coordinates from training data are converted into sequences of location tokens and appended to the corresponding text spans (e.g., an object phrase followed by patch/location tokens). This links textual mentions to specific image regions.
- Referential expressions as links: object mentions are formatted like links (Markdown-style), connecting text spans to their location-token sequences. That structure lets the model represent and generate grounded references.
- Causal language modeling objective: trained with next-token prediction so it can generate captions, ref

## Optional: Hybrid vs Vector-Only — Side-by-Side

In [ ]:
# Run the same query through both retrievers and compare what each finds
test_query = "What safety measures does LLaMA 2 have?"
query_bundle = QueryBundle(test_query)

print(f"Query: {test_query}\n")
print("=" * 60)

# Vector-only
try:
    vector_nodes = vector_retriever.retrieve(test_query)
    print(f"Vector-only ({len(vector_nodes)} nodes):")
    for i, n in enumerate(vector_nodes[:3], 1):
        score_str = f"{n.score:.4f}" if n.score is not None else "N/A"
        print(f"  [{i}] score={score_str} | {n.text[:100]}...")
except Exception as e:
    print(f"Vector retriever error: {e}")

print()

# Hybrid
try:
    hybrid_nodes = hybrid_retriever.retrieve(test_query)
    print(f"Hybrid ({len(hybrid_nodes)} nodes):")
    for i, n in enumerate(hybrid_nodes[:3], 1):
        score_str = f"{n.score:.4f}" if n.score is not None else "N/A"
        print(f"  [{i}] score={score_str} | {n.text[:100]}...")
except Exception as e:
    print(f"Hybrid retriever error: {e}")

Query: What safety measures does LLaMA 2 have?

Vector-only (6 nodes):
  [1] score=0.3321 | One quirk of sentencepiece is that when decoding a sequence, if the first token is the start of the ...
  [2] score=0.3281 | on how to use prompt tuning to adapt the LLaMA model for text classification task. 🌎<PipelineTag pip...
  [3] score=0.3197 | it doesnt degrade over time. The same principles apply to LLM-based applications  with the key diffe...

Hybrid (12 nodes):
  [1] score=0.3321 | One quirk of sentencepiece is that when decoding a sequence, if the first token is the start of the ...
  [2] score=N/A | and a concluding prompt. The input must follow a specific pattern to achieve the best results  which...
  [3] score=0.3281 | on how to use prompt tuning to adapt the LLaMA model for text classification task. 🌎<PipelineTag pip...


# Evaluate

Run the following code if you want to generate an evaluation dataset from scratch. You can choose to download an evaluation dataset running the cell after this one.

In [ ]:
# from llama_index.core.evaluation import generate_question_context_pairs

# # Create questions for each segment. These questions will be used to
# # assess whether the retriever can accurately identify and return the
# # corresponding segment when queried.
# rag_eval_dataset = generate_question_context_pairs(
#     nodes, llm=Settings.llm, num_questions_per_chunk=1
# )

# # We can save the evaluation dataset as a json file for later use.
# rag_eval_dataset.save_json("./rag_eval_dataset_question_context.json")

You can download a version of the evaluation dataset with the following code cell, so that you don't have to create the eval dataset from scratch with the code above.

In [ ]:
from huggingface_hub import hf_hub_download
from llama_index.finetuning.embeddings.common import EmbeddingQAFinetuneDataset

# Download the evaluation dataset
hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="rag_eval_dataset_question_context_subset_50.json", repo_type="dataset", local_dir=".")
rag_eval_dataset = EmbeddingQAFinetuneDataset.from_json("./rag_eval_dataset_question_context_subset_50.json")

/tmp/ipykernel_3180/9538488.py:2: DeprecationWarning: llama-index-finetuning is deprecated and no longer maintained. It will not receive any further updates.
  from llama_index.finetuning.embeddings.common import EmbeddingQAFinetuneDataset


(…)_dataset_question_context_subset_50.json: 0.00B [00:00, ?B/s]

In [ ]:
import pandas as pd

#  A simple function to show the evaluation result.
def from_eval_results_to_dataframe(name, eval_results):
    """Convert evaluation results to a pandas dataframe."""
    metric_dicts = []
    for eval_result in eval_results:
        metric_dict = eval_result.metric_vals_dict
        metric_dicts.append(metric_dict)

    full_df = pd.DataFrame(metric_dicts)

    hit_rate = full_df["hit_rate"].mean()
    mrr = full_df["mrr"].mean()

    metric_df = pd.DataFrame(
        {"Retriever Name": [name], "Hit Rate": [hit_rate], "MRR": [mrr]}
    )

    return metric_df

In [ ]:
from llama_index.core.evaluation import RetrieverEvaluator

# Evaluate retrievers at different top_k values.
for i in [2, 4, 6, 8, 10]:
    # Evaluate hybrid retriever
    vector_retriever = VectorIndexRetriever(index=vector_index, similarity_top_k=i)
    keyword_retriever = KeywordTableSimpleRetriever(index=keyword_index, num_chunks_per_query=i)
    hybrid_retriever = HybridRetriever(vector_retriever, keyword_retriever, max_retrieve=i)
    retriever_evaluator = RetrieverEvaluator.from_metric_names(
        ["mrr", "hit_rate"], retriever=hybrid_retriever
    )
    eval_results = await retriever_evaluator.aevaluate_dataset(rag_eval_dataset)
    print(from_eval_results_to_dataframe(f"Hybrid retriever top_{i}", eval_results))

    # Evaluate vector retriever
    vector_retriever = VectorIndexRetriever(index=vector_index, similarity_top_k=i)
    retriever_evaluator = RetrieverEvaluator.from_metric_names(
        ["mrr", "hit_rate"], retriever=vector_retriever
    )
    eval_results = await retriever_evaluator.aevaluate_dataset(rag_eval_dataset)
    print(from_eval_results_to_dataframe(f"Vector retriever top_{i}", eval_results))

           Retriever Name  Hit Rate    MRR
0  Hybrid retriever top_2      0.62  0.565
           Retriever Name  Hit Rate   MRR
0  Vector retriever top_2       0.6  0.57
           Retriever Name  Hit Rate       MRR
0  Hybrid retriever top_4       0.7  0.579667
           Retriever Name  Hit Rate       MRR
0  Vector retriever top_4      0.68  0.593333
           Retriever Name  Hit Rate       MRR
0  Hybrid retriever top_6      0.76  0.585889
           Retriever Name  Hit Rate       MRR
0  Vector retriever top_6       0.7  0.597333
           Retriever Name  Hit Rate       MRR
0  Hybrid retriever top_8      0.84  0.592094
           Retriever Name  Hit Rate       MRR
0  Vector retriever top_8      0.78  0.608405
            Retriever Name  Hit Rate       MRR
0  Hybrid retriever top_10      0.84  0.592094
            Retriever Name  Hit Rate       MRR
0  Vector retriever top_10       0.8  0.610627
